<a href="https://colab.research.google.com/github/jakob-steuer/chemoinformatics_course/blob/main/chemoinformatics_9_docking_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Docking — reading the result you just produced

You have docked your compounds twice, with two different programs, against two related proteins.
This notebook is about whether to believe any of it.

| step | the question |
|---|---|
| 1 | **Can the method reproduce an answer we already know?** |
| 2 | Do the two programs agree — on scores, or on poses? |
| 3 | Does your mutation actually touch the ligand? |
| 4 | Four methods, one compound: how far apart are they? |

Step 1 is a gate, not a formality. If a docking program cannot put a ligand back where the
crystallographer found it, nothing it says about your untested compounds means anything.

**What you need:** the `.zip` files SwissDock gave you. Put them in a folder in Colab's file pane
(left sidebar → upload) and name the folder in the next cell. If your jobs did not come back, the
notebook falls back to a prepared EGFR example so you can still do the analysis.

In [ ]:
# Install the packages Colab does not ship with (about a minute).
%pip install -q rdkit py3Dmol openbabel-wheel

In [ ]:
import os
import re
import glob
import json
import zipfile
import pathlib
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdMolAlign, rdMolDescriptors, rdDepictor, Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG
import py3Dmol

RDLogger.DisableLog("rdApp.*")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 11, "figure.dpi": 110})
print("ready.")

In [ ]:
# @title Plumbing — run once. Reading SwissDock archives, measuring poses, drawing them. { display-mode: "form" }

THREE_TO_ONE = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
                "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
                "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
                "TYR": "Y", "VAL": "V"}


def _tidy(mol, template):
    """A pose read back from PDBQT has lost its hydrogens into united atoms, so RDKit sees
    radicals. Reset the valence bookkeeping, then take bond orders from the known SMILES."""
    mol = Chem.RWMol(mol)
    for atom in mol.GetAtoms():
        atom.SetNoImplicit(False)
        atom.SetNumExplicitHs(0)
        atom.SetNumRadicalElectrons(0)
    mol = mol.GetMol()
    Chem.SanitizeMol(mol)
    # Only now can the hydrogens go: the reader was asked not to sanitize, and `removeHs` is
    # part of sanitization, so a pose still carries the polar hydrogens the docking file held.
    mol = Chem.RemoveHs(mol)
    return AllChem.AssignBondOrdersFromTemplate(template, mol)


def _strip_glue_atoms(pdbqt):
    """Undo AutoDock's macrocycle ring-opening, so the pose is a whole molecule again.

    When a ligand has a large flexible ring — loratadine's seven-membered one, say — the
    preparation breaks one of its bonds so the ring itself can be sampled. It retypes the two
    carbons either side of the break as `CG0` and adds a massless `G0` pseudo-atom on top of
    each. Left alone, the pose reads back as an open chain carrying two dummy atoms and matches
    no template. The two real carbons are still a bond length apart, so dropping the dummies and
    restoring the element lets the ring close on geometry.
    """
    lines, found_glue = [], False
    for line in pdbqt.read_text().splitlines(keepends=True):
        if line.startswith(("ATOM", "HETATM")):
            kind = line[77:79].strip()
            if kind.startswith("G") and not kind.startswith("CG"):
                found_glue = True
                continue                                  # massless pseudo-atom, no element
            if kind.startswith("CG"):
                found_glue = True
                line = line[:77] + "C " + line[79:]       # a real carbon the opener renamed
        lines.append(line)
    if not found_glue:
        return pdbqt
    closed = pdbqt.with_name(pdbqt.stem + "_closed.pdbqt")
    closed.write_text("".join(lines))
    return closed


def _rebuild_poses(raw, template, archive_name):
    """Rebuild every pose into a complete molecule, tolerating any that will not.

    A pose arrives without bond orders or hydrogens and `_tidy` restores them from the known
    SMILES. For some compounds that fails outright — an unusual fused heteroaromatic can come
    back in a form RDKit will not kekulize. The pose's *score* is unaffected by any of this, so
    keep the score and lose only the geometry, rather than losing the whole job to one bad pose.
    """
    poses, lost = [], 0
    for mol in raw:
        try:
            poses.append(None if mol is None else _tidy(mol, template))
        except Exception:
            poses.append(None)
            lost += 1
    if lost:
        print(f"   note: {lost} of {len(raw)} poses in {archive_name} could not be rebuilt into "
              f"a complete molecule, so they are skipped anywhere geometry is needed. Their "
              f"scores are still good. This is a quirk of reading poses back from a docking "
              f"file, not a failed run.")
    return poses


def _remark(block, key):
    """One numeric REMARK out of an Attracting Cavities pose block."""
    found = re.search(rf"REMARK {re.escape(key)}\s*:\s*(-?[\d.]+)", block)
    return float(found.group(1)) if found else None


def _element_of(atom_name):
    """Attracting Cavities writes no element column, so read it off the atom name."""
    letters = "".join(c for c in atom_name if c.isalpha()).upper()
    return {"CL": "Cl", "BR": "Br"}.get(letters[:2], letters[:1].capitalize())


def read_job(archive, max_poses=40):
    """Open one SwissDock result archive. Returns the settings, the poses and the scores.

    Handles both engines. Vina writes `vina_dock.pdbqt` with a score per MODEL; Attracting
    Cavities writes `result.dock4`, with a REMARK block per pose carrying its own score and a
    SwissParam free-energy estimate. Either way the settings come out of the archive itself, so
    nothing depends on remembering what you clicked.

    Attracting Cavities returns well over a hundred poses; `max_poses` keeps the reading quick.
    """
    archive = pathlib.Path(archive)
    folder = pathlib.Path("/content/_unpacked") / archive.stem
    if not folder.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(folder)
    files = {p.name: p for p in folder.rglob("*") if p.is_file()}

    settings = {}
    if "parameters" in files:
        for line in files["parameters"].read_text().splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                settings[key.strip()] = value.strip()

    smiles = settings.get("Ligand")
    template = Chem.MolFromSmiles(smiles) if smiles else None

    vina_file = next((files[n] for n in files
                      if n.endswith(".pdbqt") and "dock" in n.lower()), None)
    ac_file = files.get("result.dock4")

    if vina_file is not None:
        text = vina_file.read_text()
        if template is None:
            template = Chem.MolFromSmiles(
                re.search(r"REMARK SMILES ([^\s].*)", text).group(1).strip())
        scores = [float(v) for v in re.findall(r"REMARK VINA RESULT:\s+(-?\d+\.\d+)", text)]
        sdf = vina_file.with_suffix(".sdf")
        if not sdf.exists():
            source = _strip_glue_atoms(vina_file)
            os.system(f'obabel -ipdbqt "{source}" -osdf -O "{sdf}" 2>/dev/null')
        # sanitize=False here: a few compounds will not pass RDKit's aromaticity check straight
        # out of the pose file, and the supplier would raise before `_tidy` ever gets to fix it.
        poses = _rebuild_poses(list(Chem.SDMolSupplier(str(sdf), removeHs=True, sanitize=False)),
                               template, archive.name)
        extra = {}
        score_label, better = "Vina score (kcal/mol)", "lower"

    elif ac_file is not None:
        raw, scores, dgs = [], [], []
        blocks = [b for b in re.split(r"(?=^REMARK FILE_NAME)", ac_file.read_text(), flags=re.M)
                  if b.strip()]
        for block in blocks[:max_poses]:
            lines = []
            for line in block.splitlines():
                if not line.startswith("ATOM"):
                    continue
                element = _element_of(line[12:16])
                if element == "H":                      # heavy atoms only
                    continue
                lines.append(f"{line[:76]}{element:>2s}\n")
            raw.append(Chem.MolFromPDBBlock("".join(lines), removeHs=True, sanitize=False))
            scores.append(_remark(block, "MEMBER_SCORE"))
            dgs.append(_remark(block, "SP-dG"))
        poses = _rebuild_poses(raw, template, archive.name)
        extra = {"SP-dG (kcal/mol)": dgs}
        score_label, better = "AC score", "lower"

    else:
        raise FileNotFoundError(
            f"No docking poses found in {archive.name}. It contains: {sorted(files)}")

    receptor = next((files[n] for n in files if n.endswith(".pdb")
                     and "dock" not in n.lower() and "ligand" not in n.lower()), None)
    return {"name": archive.stem, "settings": settings, "smiles": smiles,
            "template": template, "poses": poses, "scores": scores, "extra": extra,
            "score_label": score_label, "better": better,
            "receptor": receptor, "folder": folder}


def fetch_pdb(pdb_id, folder="/content/structures"):
    """Download a structure from the PDB, once."""
    folder = pathlib.Path(folder); folder.mkdir(exist_ok=True, parents=True)
    target = folder / f"{pdb_id.upper()}.pdb"
    if not target.exists():
        os.system(f'wget -q -O "{target}" '
                  f'"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"')
    if target.stat().st_size < 1000:
        raise FileNotFoundError(f"could not download {pdb_id} from the PDB")
    return target


def crystal_ligand(pdb_file, resname, template):
    """Pull the crystallographic ligand out of a PDB file, as a proper molecule.

    This is the answer key for the redocking control: the position somebody measured.
    """
    lines = [l for l in pathlib.Path(pdb_file).read_text().splitlines(keepends=True)
             if l.startswith("HETATM") and l[17:20].strip() == resname]
    if not lines:
        codes = sorted({l[17:20].strip() for l in pathlib.Path(pdb_file).read_text().splitlines()
                        if l.startswith("HETATM") and l[17:20].strip() != "HOH"})
        raise ValueError(f"no residue {resname!r} in {pdb_file}. "
                         f"Ligand codes present: {codes or 'none — this is an apo structure'}")
    chain = lines[0][21]
    lines = [l for l in lines if l[21] == chain]          # one copy only
    mol = Chem.MolFromPDBBlock("".join(lines), removeHs=True)
    return AllChem.AssignBondOrdersFromTemplate(template, mol)


def rigid_core(mol):
    """Atom indices of the largest fused ring system — the part of a molecule whose position
    is actually determined. Everything else is a rotatable appendage."""
    rings = [set(r) for r in mol.GetRingInfo().AtomRings()]
    merged = []
    for ring in rings:
        joined = [ring]
        for other in merged:
            if other & ring:
                joined.append(other)
        merged = [m for m in merged if not (m & ring)] + [set().union(*joined)]
    return max(merged, key=len) if merged else set(range(mol.GetNumAtoms()))


def _best_mapping(pose, reference):
    """Which atom of the pose corresponds to which atom of the reference.

    The two molecules are the same compound but their atoms are in different orders, and a
    symmetric group (a para-substituted ring, say) can be matched more than one way. Take the
    matching that gives the smallest displacement — the same thing RDKit's CalcRMS does
    internally, but we need the mapping itself, not only the number it produces.
    """
    matches = pose.GetSubstructMatches(reference, uniquify=False, maxMatches=5000)
    if not matches:
        raise ValueError("pose and reference are not the same molecule")
    ref_xyz = reference.GetConformer().GetPositions()
    pose_xyz = pose.GetConformer().GetPositions()
    return min(matches, key=lambda m: ((pose_xyz[list(m)] - ref_xyz) ** 2).sum())


def pose_rmsd(pose, reference, atom_subset=None):
    """Symmetry-aware RMSD, optionally over a subset of the reference's atoms.

    No realignment — the molecules are compared where they actually sit in the pocket.
    `atom_subset` is given as indices into the *reference*.
    """
    if atom_subset is None:
        return rdMolAlign.CalcRMS(pose, reference)
    mapping = _best_mapping(pose, reference)
    ref_xyz = reference.GetConformer().GetPositions()
    pose_xyz = pose.GetConformer().GetPositions()
    idx = sorted(atom_subset)
    diff = pose_xyz[[mapping[i] for i in idx]] - ref_xyz[idx]
    return float(np.sqrt((diff ** 2).sum(axis=1).mean()))


def atom_deviations(pose, reference):
    """How far every atom moved, so you can see *where* a pose is wrong.

    Indexed by the reference's atoms.
    """
    mapping = _best_mapping(pose, reference)
    ref_xyz = reference.GetConformer().GetPositions()
    pose_xyz = pose.GetConformer().GetPositions()
    return np.linalg.norm(pose_xyz[list(mapping)] - ref_xyz, axis=1)


def contacts(receptor_pdb, mol, cutoff=4.5):
    """Which residues the ligand touches, and how closely."""
    conf = mol.GetConformer()
    ligand = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
    residues = {}
    for line in pathlib.Path(receptor_pdb).read_text().splitlines():
        if line.startswith("ATOM"):
            key = (line[17:20].strip(), int(line[22:26]))
            residues.setdefault(key, []).append(
                np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])]))
    rows = []
    for (name, number), atoms in residues.items():
        distance = min(np.linalg.norm(a - l) for a in atoms for l in ligand)
        if distance <= cutoff:
            rows.append({"residue": f"{THREE_TO_ONE.get(name, name)}{number}",
                         "name": name, "number": number,
                         "closest approach (Å)": round(float(distance), 2)})
    return pd.DataFrame(rows).sort_values("number").reset_index(drop=True)


def show_poses(receptor_pdb, poses, labels=None, colours=None, width=780, height=520):
    """The receptor as a cartoon, with one or more ligand poses laid over it."""
    colours = colours or ["#d1495b", "#4477aa", "#2e8b57", "#9467bd"]
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pathlib.Path(receptor_pdb).read_text(), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "#cccccc", "opacity": 0.75}})
    for i, mol in enumerate(poses):
        view.addModel(Chem.MolToMolBlock(mol), "sdf")
        view.setStyle({"model": i + 1},
                      {"stick": {"radius": 0.16, "color": colours[i % len(colours)]}})
    view.zoomTo({"model": 1})
    if labels:
        # A screen-anchored label needs an explicit `position` in pixels together with an
        # `alignment`; `screenOffset` alone leaves it clipped at the top edge, or off-screen
        # entirely for every label after the first.
        for i, text in enumerate(labels):
            view.addLabel(text.strip(),
                          {"fontSize": 14, "fontColor": "white",
                           "backgroundColor": colours[i % len(colours)],
                           "backgroundOpacity": 0.9, "borderThickness": 0.0,
                           "useScreen": True, "inFront": True, "fixed": True,
                           "alignment": "topLeft",
                           "position": {"x": 12, "y": 12 + 28 * i, "z": 0}})
    return view


def highlight_residues(view, receptor_pdb, numbers, colour="#ff8800"):
    """Draw named residues as sticks, to show where a mutation sits."""
    view.addStyle({"model": 0, "resi": [int(n) for n in numbers]},
                  {"stick": {"radius": 0.18, "color": colour}})
    return view


ROTATABLE = Chem.MolFromSmarts("[!$(*#*)&!D1]-&!@[!$(*#*)&!D1]")


def rotatable_bonds(mol):
    """Single, non-ring bonds between two non-terminal heavy atoms — every one of these is a
    torsion the docking program has to guess."""
    found = set()
    for a, b in mol.GetSubstructMatches(ROTATABLE):
        bond = mol.GetBondBetweenAtoms(a, b)
        if bond is not None:
            found.add(bond.GetIdx())
    return sorted(found)


def sketch_ligand(mol, width=640, height=360):
    """2D depiction with the rigid ring system shaded and every rotatable bond marked."""
    flat = Chem.Mol(mol)
    rdDepictor.Compute2DCoords(flat)
    core = sorted(rigid_core(flat))
    bonds = rotatable_bonds(flat)
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, flat,
        highlightAtoms=core, highlightAtomColors={i: (0.80, 0.86, 0.93) for i in core},
        highlightBonds=bonds, highlightBondColors={b: (0.82, 0.29, 0.36) for b in bonds})
    drawer.FinishDrawing()
    return SVG(drawer.GetDrawingText())


def conformer_ensemble(mol, n=12, seed=0xC0FFEE):
    """Generate n conformers and superpose them on the rigid ring system.

    Also returns, per atom, how far that atom moves across the ensemble once the core is aligned.
    That is a direct measure of which parts of the molecule have a defined position and which do
    not — computed from the molecule alone, with no protein involved.
    """
    work = Chem.AddHs(Chem.Mol(mol))
    if AllChem.EmbedMultipleConfs(work, numConfs=n, randomSeed=seed) == 0:
        raise RuntimeError("could not generate conformers for this molecule")
    AllChem.MMFFOptimizeMoleculeConfs(work, maxIters=500)
    work = Chem.RemoveHs(work)
    core = sorted(rigid_core(work))
    rdMolAlign.AlignMolConformers(work, atomIds=core)
    xyz = np.array([c.GetPositions() for c in work.GetConformers()])
    spread = np.sqrt(((xyz - xyz.mean(axis=0)) ** 2).sum(axis=2).mean(axis=0))
    return work, spread, set(core)


def show_ensemble(mol, width=760, height=420, colour="#4477aa", core=()):
    """Every conformer of an aligned ensemble, drawn over one another."""
    view = py3Dmol.view(width=width, height=height)
    for cid in range(mol.GetNumConformers()):
        view.addModel(Chem.MolToMolBlock(mol, confId=cid), "sdf")
        view.setStyle({"model": cid}, {"stick": {"radius": 0.08, "color": colour}})
        if core:
            view.addStyle({"model": cid, "serial": [i + 1 for i in sorted(core)]},
                          {"stick": {"radius": 0.16, "color": "#444444"}})
    view.zoomTo()
    return view


AROMATIC_RESIDUES = {"PHE", "TYR", "TRP", "HIS"}


def _receptor_atoms(receptor_pdb):
    """Protein atoms as (residue key, element, coordinates)."""
    out = []
    for line in pathlib.Path(receptor_pdb).read_text().splitlines():
        if line.startswith("ATOM"):
            name = line[12:16].strip()
            element = (line[76:78].strip() or name[0]).capitalize()
            out.append(((line[17:20].strip(), int(line[22:26])), name, element,
                        np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])))
    return out


def interactions(receptor_pdb, mol, hbond=3.5, apolar=4.0, stacking=5.5):
    """A crude geometric interaction fingerprint: which residues the ligand touches, and how.

    Distance criteria only — no angles, no protonation reasoning. A real analysis (PLIP, ProLIF)
    adds both. This is deliberately simple so you can read exactly what it counted, and it is
    only meant for comparing two poses of the same ligand against each other.
    """
    conf = mol.GetConformer()
    coords = conf.GetPositions()
    polar = [i for i, a in enumerate(mol.GetAtoms()) if a.GetSymbol() in ("N", "O")]
    greasy = [i for i, a in enumerate(mol.GetAtoms())
              if a.GetSymbol() in ("C", "F", "Cl", "Br", "I")]
    ring_centres = [coords[list(r)].mean(axis=0) for r in mol.GetRingInfo().AtomRings()
                    if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in r)]

    found = {}
    for key, name, element, xyz in _receptor_atoms(receptor_pdb):
        label = f"{THREE_TO_ONE.get(key[0], key[0])}{key[1]}"
        kinds = found.setdefault(label, set())
        if element in ("N", "O"):
            if any(np.linalg.norm(coords[i] - xyz) <= hbond for i in polar):
                kinds.add("H-bond")
        if element == "C":
            if any(np.linalg.norm(coords[i] - xyz) <= apolar for i in greasy):
                kinds.add("hydrophobic")
        if key[0] in AROMATIC_RESIDUES and name in ("CG", "CZ", "CE2", "CD2"):
            if any(np.linalg.norm(c - xyz) <= stacking for c in ring_centres):
                kinds.add("π-stack")
    rows = [{"residue": k, "interactions": ", ".join(sorted(v))}
            for k, v in found.items() if v]
    return pd.DataFrame(rows).sort_values("residue").reset_index(drop=True)


def compare_interactions(receptor_pdb, poses_by_label):
    """Side-by-side interaction fingerprints, so you can see where two poses differ."""
    tables = {label: interactions(receptor_pdb, mol).set_index("residue")["interactions"]
              for label, mol in poses_by_label.items()}
    merged = pd.DataFrame(tables).fillna("—")
    merged["agreement"] = ["same" if len(set(r)) == 1 else "DIFFERENT"
                           for r in merged.to_numpy()]
    return merged


def export_for_chimerax(folder, receptor_pdb, poses_by_label, highlight=()):
    """Write the poses plus a ChimeraX script that opens and colours them.

    Students visualise in ChimeraX; this saves them assembling the session by hand. Download the
    folder, then in ChimeraX:  open /path/to/view.cxc
    """
    folder = pathlib.Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    receptor_name = pathlib.Path(receptor_pdb).name
    (folder / receptor_name).write_text(pathlib.Path(receptor_pdb).read_text())

    palette = ["gold", "cornflowerblue", "mediumseagreen", "orchid"]
    lines = [f"open {receptor_name}", "hide atoms", "show cartoon",
             "color #1 gray(200)", "lighting soft"]
    for i, (label, mol) in enumerate(poses_by_label.items(), start=2):
        safe = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")
        Chem.MolToMolFile(mol, str(folder / f"{safe}.sdf"))
        lines += [f"open {safe}.sdf", f"color #{i} {palette[(i - 2) % len(palette)]}",
                  f"style #{i} stick", f"rename #{i} {safe}"]
    if highlight:
        selection = ",".join(str(int(h)) for h in highlight)
        lines += [f"show #1:{selection} atoms", f"color #1:{selection} orange",
                  f"style #1:{selection} stick", f"label #1:{selection} residues"]
    lines += ["view #2", "set bgColor white"]
    (folder / "view.cxc").write_text("\n".join(lines) + "\n")
    return folder


def reference_receptor(pdb_file, ligand_code, out="/content/reference_receptor.pdb"):
    """The protein chain the ligand actually sits in, straight from the deposited structure.

    Why not use the receptor the docking server returns? Because SwissParam rewrites it: for the
    Attracting Cavities run it comes back with no chain identifier and residues renumbered
    contiguously, against a crystal numbering that has gaps. The coordinates are untouched, so
    poses are still comparable — but residue *numbers* are not, and a contact table built on the
    rewritten file silently reports the wrong residues. Everything residue-level here is therefore
    measured against the deposited structure.
    """
    lines = pathlib.Path(pdb_file).read_text().splitlines(keepends=True)
    hetatm = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == ligand_code]
    if not hetatm:
        raise ValueError(f"no {ligand_code!r} in {pdb_file}")
    chain = hetatm[0][21]
    protein = [l for l in lines if l.startswith("ATOM") and l[21] == chain]
    out = pathlib.Path(out)
    out.write_text("".join(protein) + "END\n")
    return out


def read_boltz(folder, template, chain="B"):
    """The ligand, the affinity and the confidence out of a Boltz-2 result folder."""
    folder = pathlib.Path(folder)
    model = next(folder.rglob("*_model_0.pdb"), None)
    affinity = next(folder.rglob("affinity_*.json"), None)
    confidence = next(folder.rglob("confidence_*_model_0.json"), None)
    if model is None:
        raise FileNotFoundError(f"no *_model_0.pdb under {folder}")
    out = {"model": model,
           "affinity": json.loads(affinity.read_text()) if affinity else None,
           "confidence": json.loads(confidence.read_text()) if confidence else None}
    if out["affinity"]:
        out["pIC50"] = 6 - out["affinity"]["affinity_pred_value"]
    return out


def _ca_atoms(path, chain, offset=0):
    found = {}
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith("ATOM") and line[21] == chain and line[12:16].strip() == "CA":
            found[int(line[22:26]) + offset] = np.array(
                [float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return found


def superpose_ligand(predicted_pdb, reference_pdb, template,
                     ligand_chain="B", protein_chain="A", offset=0):
    """Put a predicted complex into the crystal's frame, and return its ligand.

    A cofolded structure comes out in an arbitrary orientation, so comparing its ligand with a
    crystal pose means first superposing the two proteins on their shared backbone. Returns the
    moved ligand and the backbone RMSD, which tells you how much of any ligand disagreement is
    really protein disagreement.
    """
    predicted = _ca_atoms(predicted_pdb, protein_chain, offset)
    reference = _ca_atoms(reference_pdb, protein_chain)
    shared = sorted(set(predicted) & set(reference))
    if len(shared) < 30:
        raise ValueError(f"only {len(shared)} residues matched — check `offset`")
    P = np.array([predicted[i] for i in shared])
    Q = np.array([reference[i] for i in shared])
    Pc, Qc = P - P.mean(0), Q - Q.mean(0)
    U, _, Vt = np.linalg.svd(Pc.T @ Qc)
    rotation = Vt.T @ np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))]) @ U.T
    backbone = float(np.sqrt(((Pc @ rotation.T - Qc) ** 2).sum(axis=1).mean()))

    lines = [l for l in pathlib.Path(predicted_pdb).read_text().splitlines()
             if l.startswith(("ATOM", "HETATM")) and l[21] == ligand_chain]
    xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in lines])
    moved = (xyz - P.mean(0)) @ rotation.T + Q.mean(0)
    block = "".join(f"{l[:30]}{moved[i][0]:8.3f}{moved[i][1]:8.3f}{moved[i][2]:8.3f}{l[54:]}\n"
                    for i, l in enumerate(lines))
    return _tidy(Chem.MolFromPDBBlock(block, removeHs=True, sanitize=False), template), backbone


print("plumbing ready.")

def target_of(job):
    """Which receptor file this job was docked into, as the archive itself recorded it."""
    return job["settings"].get("Target", "?")


def engine_of(job):
    """Which docking program produced this job, as the archive itself recorded it."""
    return job["settings"].get("Method", "?")


def receptor_chain_near(pdb_file, mol, out="/content/second_receptor.pdb"):
    """The protein chain a pose sits against, from a deposited structure.

    `reference_receptor` picks the chain by the crystallographic ligand inside it. The second
    structure of a two-target project usually has no crystal copy of your compound, so there is
    no ligand code to pick by — pick the chain the docked pose lands closest to instead.

    Deposited numbering either way, for the reason given in `reference_receptor`: never the
    receptor the docking server hands back, which returns renumbered.
    """
    ligand = mol.GetConformer().GetPositions()
    lines = pathlib.Path(pdb_file).read_text().splitlines(keepends=True)
    chains = {}
    for line in lines:
        if line.startswith("ATOM"):
            chains.setdefault(line[21], []).append(
                (float(line[30:38]), float(line[38:46]), float(line[46:54])))
    if not chains:
        raise ValueError(f"no protein chains found in {pdb_file}")
    closest = {}
    for chain, atoms in chains.items():
        xyz = np.array(atoms)
        closest[chain] = float(np.linalg.norm(xyz[:, None, :] - ligand[None, :, :],
                                              axis=-1).min())
    chain = min(closest, key=closest.get)
    out = pathlib.Path(out)
    out.write_text("".join(l for l in lines if l.startswith("ATOM") and l[21] == chain) + "END\n")
    return out


## Your files

Upload your SwissDock `.zip` files into a folder (left sidebar → Files → upload), then name that
folder below. Leave `USE_EXAMPLE` as `True` if your jobs are not back yet — the notebook will
fetch a prepared EGFR wild-type / T790M set instead, and everything downstream works the same.

In [ ]:
MY_RESULTS   = "/content/results"        # <-- folder holding your SwissDock .zip files
USE_EXAMPLE  = True                      # <-- True = use the prepared EGFR example instead

MY_DRUG      = "gefitinib"               # <-- the compound you docked
CRYSTAL_PDB  = "4WKQ"                    # <-- the structure you docked into
LIGAND_CODE  = "IRE"                     # <-- 3-letter code of ITS crystallographic ligand
MUTATED_SITE = 790                       # <-- residue number that differs between your two
                                         #     proteins (0 if you are on the selectivity track)
SECOND_PDB   = "4I22"                    # <-- part 3b only: the OTHER structure you docked
                                         #     into, if your project has two ("" if it has one)
BOLTZ_RESULTS = "/content/boltz_out"     # <-- unzipped output of notebook 9b, or "" to skip
BOLTZ_OFFSET  = 695                      # <-- add this to Boltz residue 1 to get the PDB numbering

In [ ]:
EXAMPLE_URL = ("https://raw.githubusercontent.com/jakob-steuer/chemoinformatics_course/"
               "main/docking_example/")
# One ligand in one receptor through both engines — which is what parts 1 and 2 need — plus
# the same ligand in the *other* structure, which is what part 3b needs. Jobs are grouped by the
# receptor recorded in their own archive, so the two coordinate frames stay separate: part 1
# tests only the jobs belonging to CRYSTAL_PDB and says which ones it set aside.
# The same folder on GitHub holds seven more archives (osimertinib, and a repeat run);
# `docking_example/README.md` lists them, and you can add their names here.
EXAMPLE_FILES = ["WT_gefitinib_vina.zip", "WT_gefitinib_ac.zip", "T790M_gefitinib_ac.zip"]

results = pathlib.Path(MY_RESULTS)
archives = sorted(results.glob("*.zip")) if results.exists() else []

if USE_EXAMPLE or not archives:
    if not USE_EXAMPLE:
        print(f"No .zip files found in {MY_RESULTS} — falling back to the prepared example.\n")
    results = pathlib.Path("/content/example")
    results.mkdir(exist_ok=True)
    for name in EXAMPLE_FILES:
        target = results / name
        if not target.exists():
            os.system(f'wget -q -O "{target}" "{EXAMPLE_URL}{name}"')
    archives = sorted(results.glob("*.zip"))

print(f"{len(archives)} docking job(s) found:")
for a in archives:
    print("   ", a.name)

jobs = {a.stem: read_job(a) for a in archives}

summary = pd.DataFrame([{"job": name,
                         "engine": j["settings"].get("Method", "?"),
                         "target": j["settings"].get("Target", "?"),
                         "exhaustivity": j["settings"].get("Sampling exhaustivity", "?"),
                         "box centre": j["settings"].get("Box center", "?"),
                         "poses read": len(j["poses"]),
                         "best score": round(min(j["scores"]), 2) if j["scores"] else None,
                         "score is": j["score_label"]}
                        for name, j in jobs.items()])
display(summary.set_index("job"))

Read that table before anything else. Every setting came out of the archive, not out of memory —
if two jobs you mean to compare were run with different box centres or different exhaustivity,
you will see it here, and the comparison is void until you redo one of them.

Jobs on **different receptors** are expected to have different box centres: every crystal has its
own coordinates. What must match is the settings of jobs on the *same* receptor.

## Before anything else: look at the molecule

You are about to ask two programs where this compound sits in a pocket. It is worth thirty seconds
on what you are handing them.

Two features of a drug-sized molecule decide how hard that question is:

* **The rigid ring system.** Fused rings cannot bend. Their shape is fixed before docking starts,
  so the program only has to find the right *place* and *orientation* for them.
* **The rotatable bonds.** Every one is a torsion the program has to get right as well. They
  multiply: each bond adds roughly another threefold of conformational space to search.

Below, the ring system is shaded blue and every rotatable bond is marked in red.

In [ ]:
drug = list(jobs.values())[0]["template"]      # the ligand, as RDKit read it from your archive

print(f"{MY_DRUG}: {drug.GetNumAtoms()} heavy atoms, "
      f"{len(rotatable_bonds(drug))} rotatable bonds, "
      f"MW {Descriptors.MolWt(drug):.0f}, "
      f"TPSA {rdMolDescriptors.CalcTPSA(drug):.0f} Å\u00b2, "
      f"cLogP {Descriptors.MolLogP(drug):.1f}")
display(sketch_ligand(drug))

### How much does the flexible part actually move?

That is not a rhetorical question — it can be measured, without any protein at all. Generate a
set of conformers of the free molecule, superpose them on the rigid ring system, and see how far
each atom wanders.

The dark sticks are the ring system everything is aligned on. The thin ones are every conformer
drawn on top of each other.

In [ ]:
ensemble, spread, core = conformer_ensemble(drug, n=12)
rest = sorted(set(range(ensemble.GetNumAtoms())) - core)

rows = [{"part": "rigid ring system", "atoms": len(core),
         "spread across conformers (Å)": round(float(spread[sorted(core)].mean()), 2)}]
if rest:
    rows.append({"part": "everything else", "atoms": len(rest),
                 "spread across conformers (Å)": round(float(spread[rest].mean()), 2)})
display(pd.DataFrame(rows).set_index("part"))

show_ensemble(ensemble, core=core).show()

The ring system's number is near zero **by construction** — it is what everything was aligned on.
The informative number is the other one: how far the rest of the molecule travels while the core
is held still.

Whatever spread you just measured there is roughly the accuracy you can *hope* for on the flexible
part, before the protein, the scoring function or the search have had any say.

For gefitinib that comes out at **0.01 Å for the ten-atom quinazoline against 2.1 Å for the other
twenty-one atoms** — the morpholinopropoxy tail sweeping through a couple of ångströms, with the
anilino ring swinging on its own rotatable bond as well. So when the redocking control in a moment reports a single number
for the whole molecule, you already know that number is mostly a statement about the tail — and
you already know to ask for it broken down.

# 1 — Can it find an answer we already know?

The honest test of a docking protocol is to take a ligand whose position was determined
experimentally, throw that position away, and see whether the program puts it back.

Gefitinib was crystallised in 4WKQ. We stripped it out before docking, so the program never saw
it. The crystal pose is the answer key.

**The convention is 2 Å.** Below that, the pose is considered reproduced; above it, not.

In [ ]:
structure = fetch_pdb(CRYSTAL_PDB)
receptor = reference_receptor(structure, LIGAND_CODE)

# The control only means something for jobs that docked THIS ligand into THIS structure.
# A job with a different compound cannot be compared to this answer key at all, and a job
# docked into a different crystal is in a different coordinate frame — its RMSD would be a
# large meaningless number rather than an error. So pick the comparable jobs explicitly.
reference_template = None
for j in jobs.values():
    try:
        crystal_try = crystal_ligand(structure, LIGAND_CODE, j["template"])
    except Exception:
        continue
    reference_template, crystal = j["template"], crystal_try
    break
if reference_template is None:
    raise ValueError(f"None of your jobs docked {LIGAND_CODE}, so this redocking control cannot "
                     f"run. Check MY_DRUG, CRYSTAL_PDB and LIGAND_CODE in the settings cell.")
reference_smiles = Chem.MolToSmiles(reference_template)

# How close does each job of the right compound get to the answer key?
nearest_by_job, skipped = {}, {}
for name, j in jobs.items():
    if Chem.MolToSmiles(j["template"]) != reference_smiles:
        skipped[name] = "a different compound, so there is nothing to compare it to"
        continue
    nearest = min((pose_rmsd(m, crystal) for m in j["poses"] if m is not None), default=None)
    if nearest is None:
        skipped[name] = "no readable poses"
    else:
        nearest_by_job[name] = nearest

if not nearest_by_job:
    raise ValueError(f"No job here docked {LIGAND_CODE}, so this control cannot run. Check "
                     f"MY_DRUG, CRYSTAL_PDB and LIGAND_CODE in the settings cell.")

# The answer key belongs to one receptor, and that is the one whose jobs actually land on it.
# Every other receptor is a different crystal in a different coordinate frame, so its RMSD
# against this key would be a large meaningless number rather than an error. Those jobs are
# therefore set aside by *name* — the archive records which receptor it used — and not by how
# far away they landed, which two structures in similar frames could pass by luck.
reference_target = target_of(jobs[min(nearest_by_job, key=nearest_by_job.get)])

controlled = {}
for name, nearest in nearest_by_job.items():
    if target_of(jobs[name]) != reference_target:
        skipped[name] = (f"docked into {target_of(jobs[name])}, a different crystal from the one "
                         f"this answer key comes from — part 3b compares those two")
    elif nearest > 25:
        skipped[name] = (f"closest pose is {nearest:.0f} Å from the answer key, so this did not "
                         f"dock into {CRYSTAL_PDB} at all")
    else:
        controlled[name] = jobs[name]

if not controlled:
    raise ValueError(f"No job docked this compound into {CRYSTAL_PDB}: the closest any pose came "
                     f"to the {LIGAND_CODE} answer key was {min(nearest_by_job.values()):.0f} Å. "
                     f"Check CRYSTAL_PDB and LIGAND_CODE in the settings cell.")

print(f"answer key : {LIGAND_CODE} from {CRYSTAL_PDB}, {crystal.GetNumAtoms()} heavy atoms")
print(f"receptor   : chain containing it, from the deposited structure")
print(f"control on : {len(controlled)} of {len(jobs)} job(s)")
for name, why in skipped.items():
    print(f"   skipped {name} — {why}")
print()

control, detail = [], {}
for name, j in controlled.items():
    rmsds = [None if m is None else pose_rmsd(m, crystal) for m in j["poses"]]
    usable = [(rank, r) for rank, r in enumerate(rmsds, start=1) if r is not None]
    detail[name] = pd.DataFrame({"pose": [k for k, _ in usable],
                                 j["score_label"]: [j["scores"][k - 1] for k, _ in usable],
                                 "RMSD to crystal (Å)": [round(r, 2) for _, r in usable]})
    top = usable[0][1]
    best_rank, best = min(usable, key=lambda t: t[1])
    near = sum(1 for _, r in usable if r < 2.0)
    control.append({"job": name,
                    "engine": j["settings"].get("Method", "?"),
                    "poses": len(usable),
                    "top-pose RMSD (Å)": round(top, 2),
                    "best-pose RMSD (Å)": round(best, 2),
                    "poses under 2 Å": f"{near}/{len(usable)}",
                    "exhaustivity": j["settings"].get("Sampling exhaustivity", "?")})

display(pd.DataFrame(control).set_index("job"))
figure, axes = plt.subplots(1, len(detail), figsize=(5.4 * len(detail), 4.4), squeeze=False)
for ax, (name, table) in zip(axes[0], detail.items()):
    label = [c for c in table.columns if c not in ("pose", "RMSD to crystal (Å)")][0]
    ax.scatter(table["RMSD to crystal (Å)"], table[label], s=60, color="#4477aa",
               edgecolor="white", zorder=3)
    ax.scatter(table["RMSD to crystal (Å)"].iloc[0], table[label].iloc[0], s=240, marker="*",
               color="#d1495b", edgecolor="white", zorder=4, label="best-scoring pose")
    ax.axvline(2.0, color="black", ls="--", lw=1.4)
    ax.set_xlabel("RMSD to the crystal pose (Å)")
    ax.set_ylabel(label)
    ax.set_title(controlled[name]["settings"].get("Method", name), fontsize=11)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Two different questions hide in those numbers, and the plot separates them.

**Did the search ever generate the right pose?** The lowest RMSD anywhere in the list. If even the
best pose is far away, the program never found the answer and no scoring function could have
rescued it.

**Did the scoring function recognise it?** The RMSD of the *best-scoring* pose — the one you would
actually report. A search that finds the right answer and ranks it ninth is a scoring failure,
which is a different problem with a different fix.

### How much weight will a single RMSD carry?

Very little, and this is worth being concrete about.

Running Vina on this system five times, changing nothing but the random seed, gives top-pose RMSDs
of **2.29, 2.39, 2.48, 2.50 and 4.24 Å**, with scores from −7.41 to −8.21 kcal/mol. Same inputs,
same settings, nearly 2 Å of spread.

So a single top-pose RMSD is worth about ±1 Å, and you should never rank two programs by a
difference smaller than that.

Two other things move the answer, and it is worth knowing their size relative to that noise.

**What you did to the receptor.** This structure has two residues modelled in alternate
conformations, one of them Asp855, 3.9 Å from the ligand. Keeping or discarding the second
conformation shifts the median top-pose RMSD from 2.48 Å to 2.83 Å across those same five seeds —
a real effect, but smaller than the seed noise. Preparation choices matter; they are not usually
what decides whether a redocking passes.

**Which number you are asking about.** Attracting Cavities was run twice here on byte-identical
inputs and identical settings. Everything about the ranking moved:

| | run 1 | run 2 |
|---|---|---|
| top-pose RMSD | 2.88 Å | 3.41 Å |
| best-pose RMSD | 0.86 Å | 1.84 Å |
| poses under 2 Å | 6/40 | 2/40 |
| AC score of the top pose | 2.31 | 3.81 |
| **RMSD of the ring system, top pose** | **0.31 Å** | **0.30 Å** |

Read the last row against the others. The quinazoline ring system lands in the same place to
0.01 Å, while the count of acceptable poses changes threefold and the headline RMSD moves half an
ångström. Nothing about the inputs changed; this is the method's own noise.

So the quantity to quote is **where the scaffold sits**, because that is the part the method
determines reproducibly. Pose counts, score values and whole-molecule RMSDs from a single run are
all softer than they look, and a difference of less than about 1 Å between two runs — or two
programs — is not a difference at all.

**And the two engines were not given matched budgets.** Look at the exhaustivity column — the
scales are not the same quantity, so there is no setting at which Vina and Attracting Cavities are
"equally hard tried". Any head-to-head here is partly a comparison of SwissDock's two defaults.
One ligand, one pocket, one run each is an illustration, not a benchmark; published redocking
success rates for these programs sit anywhere between 50 % and 80 % depending on the test set.

The transferable lesson is not *which program is better*. It is that **you cannot tell in advance,
so you run the control on your own system** — the same conclusion the QSAR notebook reached about
whether a model would work on a given target.

In [ ]:
# where is the pose wrong?  whole molecule, rigid core, and everything else
#   use the engine that came closest — there is no point dissecting the worse one
winner = min(controlled, key=lambda n: pose_rmsd(controlled[n]["poses"][0], crystal))
job = controlled[winner]
top_pose = job["poses"][0]
print(f"dissecting the {job['settings'].get('Method', '?')} top pose "
      f"(the closer of the two)\n")
core = rigid_core(crystal)
rest = set(range(crystal.GetNumAtoms())) - core

print(f"{'whole molecule':<28s} {crystal.GetNumAtoms():2d} atoms   "
      f"RMSD {pose_rmsd(top_pose, crystal):.2f} Å")
print(f"{'largest ring system':<28s} {len(core):2d} atoms   "
      f"RMSD {pose_rmsd(top_pose, crystal, core):.2f} Å")
print(f"{'everything else':<28s} {len(rest):2d} atoms   "
      f"RMSD {pose_rmsd(top_pose, crystal, rest):.2f} Å")

deviation = atom_deviations(top_pose, crystal)
order = np.argsort(deviation)[::-1]
figure, ax = plt.subplots(figsize=(9, 3.6))
colours = ["#4477aa" if i in core else "#d1495b" for i in order]
ax.bar(range(len(order)), deviation[order], color=colours)
ax.axhline(2.0, color="black", ls="--", lw=1.2)
ax.set_xlabel("atoms, worst first")
ax.set_ylabel("displacement (Å)")
ax.set_title("blue = in the rigid ring system, red = a rotatable appendage")
plt.tight_layout()
plt.show()

This is where a single RMSD stops being useful. In the worked example the whole molecule comes
out at **2.88 Å**, which fails the convention — but the quinazoline ring system is at **1.13 Å**,
which is essentially exact, and everything hanging off it averages **3.41 Å**.

The part that binds is right. The parts that rotate freely — the morpholine tail reaching into
solvent above all — are wrong, and since twenty-one of the thirty-one atoms are in them, they
drag the summary number over the threshold.

(Those three numbers are not independent: square them, weight by atom count, average and take the
root, and you get 2.88 Å back. That is worth checking once, because it is what makes the split a
decomposition of the RMSD rather than three unrelated statistics.)

That matters for what you do next. The tail is not what makes gefitinib an EGFR inhibitor; the
quinazoline that hydrogen-bonds to the hinge is. A pose that gets the core right is usable for
asking which residues the drug touches, even though it failed the formal test.

In [ ]:
view = show_poses(receptor, [crystal, top_pose],
                  labels=["  crystal pose (the answer)", "  best-scoring docked pose"])
view.show()

### Exercise 1 — Is your protocol allowed to continue?

Answer these three before you look at a single result for an untested compound.

1. **Did the search find the crystal pose?** Lowest RMSD anywhere in your list — above or below 2 Å?
2. **Did the score recognise it?** RMSD of your best-scoring pose. Is it the same pose?
3. **Where is it wrong?** Core or periphery? Look at the bar chart, not just the number.

<details>
<summary>🔍 What the worked example shows, and what to do about it</summary>

Measured across repeated runs of this one system, rather than from a single submission:

| | top-pose RMSD | best pose | poses under 2 Å | ring system, top pose |
|---|---|---|---|---|
| Vina → 4WKQ, two runs | 2.74, 2.88 Å | 2.74, 2.88 Å | **0/20, 0/20** | 1.00, 1.13 Å |
| Vina → 4I22, one run | 3.45 Å | 3.45 Å | **0/20** | 0.75 Å |
| Attracting Cavities → 4WKQ, two identical runs | 2.88, 3.41 Å | 0.86, 1.84 Å | 6/40, 2/40 | 0.31, 0.30 Å |
| Attracting Cavities → 4I22, one run | **1.45 Å** | 1.35 Å | 5/40 | 0.35 Å |

The engines fail in genuinely different ways here, and the last column is the only one steady
enough to carry an argument on its own.

**Vina never generates a pose within 2 Å** — nought out of twenty, three times over, sixty poses
in all. Two standalone Vina runs at twice and at sixteen times SwissDock's sampling still put the
top pose at 2.39 and 2.60 Å, so more searching did not rescue it. That is a search not reaching
the answer, and no scoring function can promote a pose that was never generated.

**Attracting Cavities generates it in every run** — two to six poses in forty — and ranks it first
only sometimes. In the first 4WKQ run the closest pose is 0.86 Å, lying at rank 15, with the first
pose under 2 Å at rank 4; the pose the method actually reports is the 2.88 Å one. Different
problem, different fix: Vina needs better sampling or better receptor preparation; AC needs a
better way to choose among poses it already has.

**And the verdict flips between the two structures — but it is the ranking that flips, not the
search.** AC's top pose passes the 2 Å test in 4I22, at 1.45 Å, and fails it in 4WKQ, at 2.88 and
3.41 Å. Read along the rows, though, and the search did at least as well in 4WKQ: closest pose
0.86 Å there against 1.35 Å in 4I22, and a comparable handful under 2 Å in both. What differs is
whether a good pose came out on top. In 4I22 the first-ranked pose is already under 2 Å; in 4WKQ
the first one that is sits at rank 4 in one run and rank 12 in the other. So the pass/fail badge
turns on the single least reproducible quantity in the table — which is the next paragraph's point.

Underneath that there is a real difference between the two pockets, and it is in the periphery
rather than the scaffold: the ring system lands at 0.3 Å in both, while the flexible tail is 1.7 Å
out in 4I22 against 3.5 and 4.1 Å in 4WKQ. One run on 4I22 is not enough to call it. It makes the
practical point anyway — whatever makes a pocket easy or hard is a property of that pocket, not of
the program, so a control that passed on somebody else's system tells you nothing about yours.

A caution about reading too much into either AC row. The two 4WKQ runs above differ in nothing at
all — same receptor file, same settings, two submissions — and they still disagree by 0.53 Å on the
top pose and by a factor of three on the pose count. A third run, made earlier with the alternate
side-chain conformations left in the receptor, gave 0.92 Å. It is tempting to read that as "keeping
altlocs helps", and it may, but the run-to-run spread above is wide enough that one run cannot
show it. Their `parameters` file would not have warned you either: it records the target's
**filename**, not its contents. If you repeat a job to estimate variability, checksum the inputs.

None of this is a verdict on the programs in general. It is a verdict on this protocol, on this
complex, which is all a control can ever give you — and all you need in order to decide what to do
next.

Formally, the wild-type redocking fails. Practically, the quinazoline ring system is at 0.3–1.1 Å
and the failure is in the rotatable periphery. The honest conclusion is the middle one:

> This protocol places the scaffold correctly to about 2 Å and does not determine the
> conformation of the flexible tail. Conclusions that depend on the scaffold's position — which
> residues are contacted, whether a mutation clashes — are supportable. Conclusions that depend
> on the tail are not.

Writing that sentence *before* looking at your results is what keeps the rest of the analysis
honest. If your own redocking is worse than about 3 Å even in the core, say so on your slide and
treat every score below as a ranking hint at best.

**What would make it better, if you had time:** a flexible receptor, retained structural waters,
or a different protonation state for the ligand. All three are real effects and all three are
outside what a one-hour exercise can do — which is itself worth knowing.

</details>

## The same test, for the AI model

If you ran the cofolding notebook, you have a third prediction of where this ligand sits — made
without any docking at all, by a model that learned from every structure in the PDB.

It can take the same exam. One extra step first: a cofolded complex comes out in an arbitrary
orientation, so the two proteins have to be superposed before their ligands can be compared. The
backbone RMSD from that superposition is worth reading on its own — if the protein is predicted
badly, a ligand disagreement may be the protein's fault rather than the ligand's.

In [ ]:
# If you did not run the cofolding notebook, fall back to the worked example — the same way the
# docking archives do. This analysis reads only three small files out of a Boltz result folder.
BOLTZ_EXAMPLE = ["gefitinib_egfr_wt_model_0.pdb",
                 "affinity_gefitinib_egfr_wt.json",
                 "confidence_gefitinib_egfr_wt_model_0.json"]

boltz_folder = pathlib.Path(BOLTZ_RESULTS) if BOLTZ_RESULTS else None
if (boltz_folder is None or not boltz_folder.exists()) and USE_EXAMPLE:
    boltz_folder = pathlib.Path("/content/example_boltz")
    boltz_folder.mkdir(exist_ok=True)
    for name in BOLTZ_EXAMPLE:
        target = boltz_folder / name
        if not target.exists():
            os.system(f'wget -q -O "{target}" "{EXAMPLE_URL}boltz_example/{name}"')
    if not any(boltz_folder.glob("*_model_0.pdb")):
        boltz_folder = None
    else:
        print("Using the prepared cofolding example (gefitinib + EGFR wild type).\n")

if boltz_folder is not None and boltz_folder.exists():
    boltz = read_boltz(boltz_folder, job["template"])
    predicted, backbone = superpose_ligand(boltz["model"], receptor, job["template"],
                                           offset=BOLTZ_OFFSET)
    print(f"backbone RMSD after superposition : {backbone:.2f} Å")
    print(f"ligand RMSD to the crystal pose   : {pose_rmsd(predicted, crystal):.2f} Å")
    if boltz["confidence"]:
        c = boltz["confidence"]
        print(f"\nthe model's own confidence: ipTM {c['iptm']:.2f}, "
              f"pLDDT {c['complex_plddt']:.2f}")
    if boltz.get("pIC50"):
        print(f"its predicted pIC50: {boltz['pIC50']:.2f}")

    comparison = pd.DataFrame([
        {"method": "cofolding (Boltz-2)", "RMSD to crystal (Å)": round(pose_rmsd(predicted, crystal), 2)},
        *[{"method": f"{j['settings'].get('Method', '?')} (docking)",
           "RMSD to crystal (Å)": round(pose_rmsd(j["poses"][0], crystal), 2)}
          for j in controlled.values() if j["poses"] and j["poses"][0] is not None],
    ]).sort_values("RMSD to crystal (Å)")
    display(comparison.set_index("method"))

    view = show_poses(receptor, [crystal, predicted],
                      labels=["  crystal pose", "  cofolded pose"])
    view.show()
else:
    print("No cofolding result available. Run the Boltz notebook and point BOLTZ_RESULTS at the\n"
          "unzipped output folder, or set USE_EXAMPLE = True to use the prepared one.")

In the worked example the order comes out like this:

| method | ligand RMSD to the crystal pose |
|---|---|
| cofolding (Boltz-2) | **0.66 Å** |
| Attracting Cavities | 2.88 Å |
| AutoDock Vina | 2.88 Å |

with the predicted backbone itself 0.50 Å from the crystal, so the protein is essentially right
and the ligand comparison is fair.

The two docking engines landing on the same number is a coincidence, and part 1 showed they
got there differently: Vina never generated a better pose, while Attracting Cavities generated
one at 0.86 Å and ranked it fifteenth. The cofolding prediction beats both by a factor of four —
with no receptor structure, no box and no search.

**Read that against the affinity.** The same Boltz run predicted pIC₅₀ **6.83** where the
measured value is **7.89**.

That gap is bigger than it looks, because pIC₅₀ is a *logarithmic* scale — it is −log₁₀ of the
molar concentration that gives half-inhibition. One unit is a **tenfold** change in how much
compound you need, so being 1.06 units low means predicting a drug that needs about **eleven times
more material** to do the same job (10<sup>1.06</sup> ≈ 11). Nothing on this axis is ever off "by
about one", which is also why the comparison plot at the end of the notebook keeps every pIC₅₀ on
a single axis and refuses to put kcal/mol beside them.

So the method that is *worst* at the number is *best* at the picture, while docking — the method
you might have trusted by reputation — is four times further out on the picture and offers no
number on this scale at all. Being good at one of these tasks says very little about being good
at the other.

**And then the obvious objection.** Gefitinib bound to EGFR has been in the PDB since 2004, and
Boltz-2 was trained on the PDB. Asking it to place gefitinib in EGFR is asking it to recall
something it has seen — the same problem notebook 7 found when a QSAR model predicted a drug that
was sitting in its own training set, and the same reason this exercise started with a redocking
control rather than a confident number.

A fair test would be a complex deposited after the model's training cut-off. That is not a thing
you can arrange for your own project compound this week, so the honest move is to say the
limitation out loud rather than quote 0.66 Å as if it were a validation.

# 2 — Two programs, one pocket

You ran the same ligand against the same receptor with **Attracting Cavities** and with
**AutoDock Vina**. They use different search algorithms and different scoring functions, so they
are two independent attempts at the same question.

Agreement between them is weak evidence that the answer is right. Disagreement is strong evidence
that at least one is wrong — and tells you not to trust a single number from either.

Two kinds of agreement, and they are not the same thing:

* **Score agreement** — do they rank the compounds in the same order? This is what you would use
  for screening.
* **Pose agreement** — do they put the molecule in the same place? This is what you need before
  you say anything about contacts or clashes.

A pair of programs can agree on scores while disagreeing on poses, which should worry you more
than the reverse.

In [ ]:
# only jobs that passed the comparability check in part 1 — two different compounds, or two
# different coordinate frames, cannot be compared pose-to-pose
engines = {}
for name, j in controlled.items():
    engine = j["settings"].get("Method", "?")
    target = j["settings"].get("Target", "?")
    engines.setdefault((target, engine), j)

print("what you have:")
for (target, engine), j in engines.items():
    print(f"   {engine:<22s} on {target:<28s} best score {min(j['scores']):.2f}")

pairs = {}
for (target, engine), j in engines.items():
    pairs.setdefault(target, {})[engine] = j
comparable = {t: d for t, d in pairs.items() if len(d) > 1}

if not comparable:
    print("\nOnly one engine per target here, so there is nothing to compare yet.")
    print("Run the same receptor through the other engine and re-run this cell.")
else:
    rows = []
    for target, by_engine in comparable.items():
        names = list(by_engine)
        for a, b in itertools.combinations(names, 2):
            ja, jb = by_engine[a], by_engine[b]
            rmsd = pose_rmsd(jb["poses"][0], ja["poses"][0])
            rows.append({"target": target,
                         f"{a} ({ja['score_label']})": round(min(ja["scores"]), 2),
                         f"{b} ({jb['score_label']})": round(min(jb["scores"]), 2),
                         "pose RMSD between them (Å)": round(rmsd, 2),
                         "same pose?": "yes" if rmsd < 2.0 else "NO"})
    display(pd.DataFrame(rows))
    print("\nThe two score columns are different quantities — the header says which. Vina's is a")
    print("kcal/mol estimate; the AC score is its own cluster score (its SP-dG, further down, is")
    print("in kcal/mol too). Compare the *order* they put compounds in, and compare the poses.")

**The scores are not comparable between programs.** Vina's number is a kcal/mol estimate; the
Attracting Cavities score is its own quantity. Putting them side by side and reading one as
"better" is the single most common mistake made with docking output. What you can compare is the
*order* they put your compounds in, and where they put the molecule in space.

In [ ]:
if comparable:
    target = list(comparable)[0]
    by_engine = comparable[target]
    names = list(by_engine)
    view = show_poses(receptor,
                      [by_engine[n]["poses"][0] for n in names],
                      labels=[f"  {n} top pose" for n in names])
    view.show()
else:
    print("nothing to overlay yet — one engine only")

### Exercise 2 — Which do you believe?

If the two top poses are more than 2 Å apart, at least one is wrong and you cannot tell which from
the scores alone. Three things that *can* break the tie:

1. **The redocking control from part 1.** Whichever engine reproduced the crystal pose better has
   earned more trust in this pocket. That is the only evidence here that is grounded in experiment.
2. **Chemical sense.** Does the pose make the hydrogen bond the medicinal chemistry literature
   says this class makes? For 4-anilinoquinazolines that is the quinazoline N1 to the hinge
   backbone NH of Met793. A pose that misses it is suspect however well it scores.
3. **Consistency across your shortlist.** A pose family that recurs for several related compounds
   is more plausible than a one-off.

<details>
<summary>🔍 Why the scores cannot settle it</summary>

Both scoring functions were fitted to reproduce measured affinities across a large, diverse set of
complexes. Neither was fitted to identify the correct pose for *your* complex. A function can be
decent at the first job and poor at the second, and routinely is — which is exactly what the
scatter in part 1 shows when good scores appear at bad RMSDs.

This is also why the redocking control comes first. It is the only measurement in the whole
exercise that has a known right answer.

</details>

## Do they agree on the chemistry, or only on the coordinates?

Two poses 1 Å apart can still make different contacts, and two poses 3 Å apart can make the same
ones. What a medicinal chemist acts on is the interaction pattern, so compare that directly.

The detector below is deliberately crude — distance cut-offs only, no angles, no protonation
reasoning. Read it as "which residues are close enough to be doing something", not as a
measurement. For anything you put in a report, PLIP or ProLIF do the job properly.

In [ ]:
poses_by_label = {}
for name, j in controlled.items():
    if j["poses"] and j["poses"][0] is not None:
        # label by engine where that is unambiguous, otherwise by job, so two jobs run with the
        # same engine cannot overwrite each other
        engine = j["settings"].get("Method", name)
        label = engine if sum(1 for k in controlled.values()
                              if k["settings"].get("Method") == engine) == 1 else name
        poses_by_label[label] = j["poses"][0]
poses_by_label["crystal"] = crystal
# `predicted` exists only if a cofolding result was actually read further up — which includes
# the prepared example, so this cannot be a test of BOLTZ_RESULTS itself.
if "predicted" in globals():
    poses_by_label["cofolded"] = predicted

fingerprints = compare_interactions(receptor, poses_by_label)
print(f"{len(fingerprints)} residues involved across all poses; "
      f"{(fingerprints['agreement'] == 'DIFFERENT').sum()} are contacted differently\n")
display(fingerprints)

The rows marked `DIFFERENT` are where your two methods actually disagree about the binding mode.
Those are the ones to look at in the viewer — and if one of them is a residue the literature says
matters for this chemotype, you have a reason to prefer one pose over the other that does not
depend on either scoring function.

In [ ]:
# every pose on top of the crystal, in one view
view = show_poses(receptor, list(poses_by_label.values()),
                  labels=[f"  {k}" for k in poses_by_label])
if MUTATED_SITE:
    highlight_residues(view, receptor, [MUTATED_SITE])
view.show()

## The energy terms behind the score

A docking score is a sum. Attracting Cavities reports the parts, which is more informative than
the total: a pose held together by nonpolar burial is a different hypothesis from one held by
polar contacts, even at the same score.

In [ ]:
for name, j in jobs.items():
    if not j.get("extra"):
        continue
    engine = j["settings"].get("Method", name)
    terms = pd.DataFrame({j["score_label"]: j["scores"], **j["extra"]})
    print(f"--- {engine}: the first few poses")
    display(terms.head(6).round(2))

    figure, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(terms[j["score_label"]], terms[list(j["extra"])[0]],
               s=55, color="#4477aa", edgecolor="white")
    ax.set_xlabel(j["score_label"])
    ax.set_ylabel(list(j["extra"])[0])
    ax.set_title(f"{engine}: do its two numbers even agree with each other?")
    plt.tight_layout()
    plt.show()

if not any(j.get("extra") for j in jobs.values()):
    print("No per-term breakdown in these archives (Vina reports only INTER and INTRA).")

Attracting Cavities reports two numbers per pose: its own cluster score and **SP-dG**, a
SwissParam free-energy estimate in kcal/mol. They do not have to agree, and when they rank poses
differently you have to decide which one you are quoting — a question most people never notice
they are answering.

Note that SP-dG *is* in kcal/mol, like Vina's score. That makes the units comparable, which is not
the same as making the numbers comparable: they come from different functions fitted to different
data. Compare the two engines by the *order* they put compounds in, and by their poses.

## Taking it to ChimeraX

You will do the real looking in ChimeraX. This writes the receptor, every pose, and a script that
opens and colours them, so you do not have to assemble the session by hand.

In [ ]:
out = export_for_chimerax("/content/chimerax_session", receptor, poses_by_label,
                          highlight=[MUTATED_SITE] if MUTATED_SITE else ())
print("written to", out)
for f in sorted(out.iterdir()):
    print("   ", f.name)
print("\nDownload the folder, then in ChimeraX:   open <folder>/view.cxc")
print("\nUseful commands once it is open:")
print("   contacts #2 restrict #1 reveal true     # what the pose touches")
print("   hbonds #2 restrict #1 reveal true       # hydrogen bonds, done properly")
print("   surface #1 enclose #1                   # the pocket as a surface")

# 3 — Does your mutation touch the drug?

For the resistance track this is the central question of the whole project. A mutation can stop a
drug working in several different ways, and they are not equally easy to see:

| mechanism | visible in a docked pose? |
|---|---|
| the new side chain collides with the ligand | **yes** — that is what this cell looks for |
| the new side chain is nowhere near the ligand | **yes, as an absence** — and that is a result |
| the mutation shifts the protein's conformational preference | no |
| the mutation changes ATP affinity rather than drug affinity | no |

The first two you can answer today. The last two need the kind of evidence this notebook cannot
produce, and saying so is part of the answer.

In [ ]:
pocket = contacts(receptor, top_pose, cutoff=4.5)
print(f"{len(pocket)} residues within 4.5 Å of the docked ligand\n")
display(pocket[["residue", "closest approach (Å)"]].set_index("residue").T)

if MUTATED_SITE:
    hit = pocket[pocket["number"] == MUTATED_SITE]
    print()
    if len(hit):
        row = hit.iloc[0]
        print(f"*** residue {MUTATED_SITE} ({row['name']}) IS in contact — "
              f"closest approach {row['closest approach (Å)']} Å")
        print("    The mutation is in the binding site. A steric or electronic explanation is")
        print("    available to you, and the pose is the evidence for it.")
    else:
        print(f"*** residue {MUTATED_SITE} is NOT within 4.5 Å of the ligand")
        print("    Whatever this mutation does, it is not a direct clash with the drug.")
        print("    That is a finding, not a failure — and it rules out the easy explanation.")

In [ ]:
if MUTATED_SITE:
    view = show_poses(receptor, [top_pose], labels=[f"  {MY_DRUG}, docked"])
    highlight_residues(view, receptor, [MUTATED_SITE])
    view.show()
    print(f"orange = residue {MUTATED_SITE}")

In the worked example, **Thr790 sits 3.4 Å from gefitinib** — the gatekeeper is in direct van der
Waals contact with the drug. Replacing that threonine with the bulkier methionine of T790M puts
more atoms into a space that was already tight, which is the textbook steric account of gefitinib
resistance.

Be careful about how much that picture proves. It is consistent with the steric story; it is not
a measurement of it. T790M also *increases* the kinase's affinity for ATP, so part of the clinical
effect is competition the pose cannot show you. A contact map answers "could this be steric?",
not "is this the whole mechanism?".

### Exercise 3 — Your own mutation

1. Is your mutated residue in the contact list? At what distance?
2. If yes: is the change bigger, smaller, differently charged? Which part of your ligand is it
   nearest?
3. If no: what else could the mutation be doing, and what evidence would you need to test it?
4. **Dock into both structures and compare the contact lists.** Does the pose move, or just the
   score?

<details>
<summary>🔍 For the selectivity track</summary>

Set `MUTATED_SITE = 0` and ask the question differently. Dock your compound into both proteins and
compare the two contact lists. The residues that appear in one and not the other are the
structural difference your selectivity has to come from. Then check them in the sequence alignment:
a contact residue that differs between the two proteins is a candidate explanation; a selectivity
difference with an *identical* contact set has to be explained by something other than the pocket
lining — shape, dynamics, or the conformation each protein prefers.

</details>

# 3b — The same compound in two pockets

If your project is **one compound against two proteins** — a drug and its resistant mutant, or a
drug and an off-target you want to avoid — then this is the comparison the whole project is
about, and it needs care, because the two structures are two different crystals.

**You cannot compare the poses themselves.** Two crystals sit in two different coordinate
frames, so an RMSD between a pose in one and a pose in the other is a meaningless number. Part 1
set those jobs aside for exactly this reason.

**What you can compare is which residues each pose touches.** Contacts are measured inside each
structure separately, so each one is in its own frame and no comparison across frames is needed.
The residues that appear in one pocket and not the other are the structural difference your
result has to come from.

The cell uses the **same engine on both** receptors wherever your set allows it. Part 2 measured
how far apart two different programs put the same molecule in the same pocket, and that gap is
usually bigger than the difference between the two pockets — so mixing engines here would mostly
measure the engines.

Set `SECOND_PDB` in the settings cell to the other structure's PDB code.

In [ ]:
# Group every job by the receptor it was docked into, as recorded in its own archive, so that
# two crystals can never be mixed up by accident.
by_target = {}
for name, j in jobs.items():
    by_target.setdefault(target_of(j), {})[name] = j

print(f"{len(by_target)} receptor(s) in this set:")
for t, group in by_target.items():
    kinds = ", ".join(sorted({g["settings"].get("Method", "?") for g in group.values()}))
    print(f"   {t:<26s} {len(group)} job(s)  {kinds}"
          + ("   <- the one part 1 tested" if t == reference_target else ""))

elsewhere = [t for t in by_target if t != reference_target]
differs = []

if not elsewhere:
    print("\nOnly one receptor here, so there is nothing to compare — this section is for a "
          "project\nwith one compound and two proteins. To run it: dock your compound into "
          "both, put all\nthe archives in the same folder, and set SECOND_PDB.")
elif not SECOND_PDB:
    print(f"\nThere are poses in {elsewhere[0]} too, but SECOND_PDB is empty. Set it to that "
          f"structure's\nPDB code in the settings cell to compare the two pockets.")
else:
    # Compare like with like. Part 2 measured how far apart two *programs* put the same molecule
    # in the same pocket, and that is usually larger than the difference between the pockets —
    # so use one engine on both receptors wherever the set allows it, preferring the engine that
    # came closest in part 1.
    here_group, there_group = by_target[reference_target], by_target[elsewhere[0]]
    shared_engines = ({engine_of(g) for g in here_group.values()}
                      & {engine_of(g) for g in there_group.values()})
    if shared_engines:
        wanted = engine_of(job) if engine_of(job) in shared_engines else sorted(shared_engines)[0]
        here_name = next(n for n, g in here_group.items() if engine_of(g) == wanted)
        there_name = next(n for n, g in there_group.items() if engine_of(g) == wanted)
    else:
        here_name, there_name = next(iter(here_group)), next(iter(there_group))
        print("\n!! No engine was run on both receptors, so the two poses below come from "
              "different\n!! programs. Part 2 showed how far apart two engines put the same "
              "molecule — expect some\n!! of the difference you are about to see to be that, "
              "rather than the two pockets.")

    here_pose = here_group[here_name]["poses"][0] if here_group[here_name]["poses"] else None
    other_pose = there_group[there_name]["poses"][0] if there_group[there_name]["poses"] else None
    if here_pose is None or other_pose is None:
        raise ValueError("a top pose could not be read, so the two pockets cannot be compared")

    # Its own deposited structure, so residue numbers are the crystallographer's. No
    # crystallographic ligand is needed: the chain is chosen by which one the pose sits against.
    other_receptor = receptor_chain_near(fetch_pdb(SECOND_PDB), other_pose)
    print(f"\n{CRYSTAL_PDB}: {engine_of(here_group[here_name])} top pose      "
          f"{SECOND_PDB}: {engine_of(there_group[there_name])} top pose")
    print(f"part 1 put the {CRYSTAL_PDB} pose {pose_rmsd(here_pose, crystal):.2f} Å from the "
          f"crystal answer — read\nthe differences below with that much slack in mind.\n")

    # Contacts are measured inside each structure separately, each in its own frame.
    here_by_number  = {int(r["number"]): r for _, r in
                       contacts(receptor, here_pose, cutoff=4.5).iterrows()}
    there_by_number = {int(r["number"]): r for _, r in
                       contacts(other_receptor, other_pose, cutoff=4.5).iterrows()}

    rows, shared, same_name = [], 0, 0
    for number in sorted(set(here_by_number) | set(there_by_number)):
        a, b = here_by_number.get(number), there_by_number.get(number)
        if a is not None and b is not None:
            shared += 1
            same_name += int(a["name"] == b["name"])
            touched, label = "both", a["residue"]
            if a["name"] != b["name"]:
                label = f"{a['residue']}/{b['residue']}"      # not the same amino acid
        else:
            differs.append(number)
            touched = CRYSTAL_PDB if a is not None else SECOND_PDB
            label = (a if a is not None else b)["residue"]
        rows.append({"residue": label,
                     f"{CRYSTAL_PDB} (Å)": a["closest approach (Å)"] if a is not None else "—",
                     f"{SECOND_PDB} (Å)": b["closest approach (Å)"] if b is not None else "—",
                     "touched in": touched})
    display(pd.DataFrame(rows).set_index("residue"))

    print(f"\n{len(differs)} of {len(rows)} pocket residues are touched in only one of the two "
          f"structures.")
    if differs:
        print(f"   {differs}")
    print(f"\nSanity check on the numbering: of the {shared} residues touched in both, "
          f"{same_name} are the\nsame amino acid in both structures.")
    if shared and same_name < shared - 2:
        print("   Mostly different, so these two structures do NOT share a residue numbering —")
        print("   they are different proteins. Lining the rows up by number is then wrong: pair")
        print("   them through a sequence alignment yourself before reading anything into it.")
    else:
        print("   Nearly all the same, so the two structures share a numbering and the rows above")
        print("   line up correctly.")
    if MUTATED_SITE and MUTATED_SITE in set(here_by_number) | set(there_by_number):
        print(f"\n*** residue {MUTATED_SITE} is in the pocket of both — the row above says how "
              f"close, in each.")

In [ ]:
if differs:
    view = show_poses(other_receptor, [other_pose],
                      labels=[f"  {MY_DRUG} in {SECOND_PDB}"])
    highlight_residues(view, other_receptor, differs[:10])
    view.show()
    print(f"orange = residues touched in only one of the two structures (up to ten shown)")

Read the `touched in` column, not the distances.

A residue touched in **both** pockets is part of the binding story in both, and cannot by itself
explain a difference between them. A residue touched in **only one** is a candidate explanation —
and if your two structures are the same protein, a residue that appears in only one is usually
telling you the pose moved, not that the pocket changed.

<details>
<summary>🔍 Reading it on the resistance track</summary>

Two structures of the same protein, differing at one position. Both contact lists should look
almost identical, and the interesting question is whether `MUTATED_SITE` appears in both.

If it does, the mutation is in the pocket in both forms, and the comparison worth making is the
distance: does the drug sit further away in the mutant, as a clash would predict? Be careful how
much that proves. The two numbers come from different crystals with different resolutions, and
part 1 measured this method's own noise at around 1 Å — so a difference smaller than that is not
a difference.

If the contact lists differ in *many* places, the likelier explanation is that the pose moved
between the two runs, which part 1 told you to expect. Compare the ring systems rather than the
whole molecules before concluding anything about the pocket.

</details>

<details>
<summary>🔍 Reading it on the selectivity track</summary>

Two genuinely different proteins. Residue numbers then mean different things in each, which is
what the sanity check at the bottom of the cell is for — if most residues touched in both are
*different* amino acids, the numbering does not line up and the rows are paired wrongly.

The honest version needs a sequence alignment: align the two proteins, then compare the residues
at equivalent *aligned* positions rather than at equal numbers. The list above is still useful as
the set of positions to look up in that alignment.

What you are looking for is a contact residue that differs in identity between the two proteins.
That is a candidate explanation for selectivity. And if every contact residue is identical, then
the selectivity has to come from somewhere a single pose cannot show you — the shape of the
pocket further out, its flexibility, or which conformation each protein prefers.

</details>

# 4 — Four methods, one compound

Across three notebooks you have now estimated the same thing four ways. Each knows something the
others do not.

| method | what it is based on | what it cannot see |
|---|---|---|
| **measured pChEMBL** | somebody's experiment | whether the assay matches your question |
| **QSAR** (notebook 7) | which substructures correlate with activity | the protein, at all |
| **docking** (today) | a physical model of the pocket | entropy, water, flexibility |
| **cofolding** (Boltz-2) | patterns across known structures | whether it has simply memorised your drug |

Fill in what you have. Anything you did not run, leave as `None`.

In [ ]:
# one row per compound; use None for anything you did not run
COMPARISON = {
    #  compound        measured   QSAR    Vina     AC      Boltz pIC50
    "gefitinib":     dict(measured=7.89, qsar=7.90, vina=-7.75, ac=None, boltz=6.83),
    # "osimertinib": dict(measured=7.43, qsar=7.26, vina=None,  ac=None, boltz=None),
}

table = pd.DataFrame(COMPARISON).T
table.columns = ["measured pIC50", "QSAR pIC50", "Vina (kcal/mol)",
                 "AC score", "Boltz pIC50"]
display(table)

# only the pIC50-scale columns can share an axis
scale = table[["measured pIC50", "QSAR pIC50", "Boltz pIC50"]].dropna(axis=1, how="all")
if len(scale.columns) > 1 and scale.notna().any().any():
    figure, ax = plt.subplots(figsize=(8, 4.2))
    scale.plot.bar(ax=ax, rot=0, width=0.75,
                   color=["#333333", "#4477aa", "#d1495b"][:len(scale.columns)])
    ax.set_ylabel("pIC50  (higher = stronger)")
    ax.set_ylim(4, 11)
    ax.set_title("the three estimates that share a scale")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
    print("Docking scores are deliberately not on this plot — kcal/mol from a scoring function")
    print("is not a pIC50 and putting them on one axis would be the mistake this week is about.")

For gefitinib on wild-type EGFR the three comparable numbers are **7.89 measured**, **7.90 from
QSAR** and **6.83 from Boltz-2**. The QSAR number looks extraordinary until you remember what
notebook 7 proved: gefitinib is in its training data, and removing it and its analogues moved the
prediction. Boltz is out by 1.06 log units, about elevenfold, on a drug it has almost certainly
also seen.

The docking score is not on the plot on purpose. −7.75 kcal/mol is not a pIC₅₀, and the fact that
it looks numerically close to 7.89 is a coincidence of units that has fooled a great many people.

### Exercise 4 — The three sentences for your slide

You have a presentation next week. Write these now, while the numbers are in front of you.

1. **What you found.** The claim, with a number, and which method produced it.
2. **What validates it.** Your redocking RMSD, and what it licenses you to say.
3. **What would change your mind.** The specific observation that would overturn it.

<details>
<summary>🔍 A worked version</summary>

> Gefitinib docks into wild-type EGFR with a Vina score of −7.75 kcal/mol, in a pose whose
> quinazoline ring system reproduces the crystallographic position to 1.13 Å, contacting Thr790
> at 3.4 Å. The redocking control places the whole molecule at 2.88 Å — outside the 2 Å convention, with the
> error confined to the solvent-exposed morpholine tail — so we treat the scaffold position as
> reliable and make no claim about the tail. The steric account of T790M resistance is consistent
> with this contact, but not demonstrated by it: T790M also raises the kinase's ATP affinity, which
> no pose can show. We would abandon the steric explanation if docking into the mutant structure
> produced an essentially unchanged pose and score, which would mean the clash we are invoking is
> not there.

Three features worth copying: every claim carries the measurement that supports it, the validation
is stated before the conclusion, and the falsifier is a specific result rather than a hedge.

</details>

# What you would do next

This notebook measured one thing: whether to believe a pose. In a real project that is where the
work starts, not where it ends.

Before anything else, notice that *"can I trust this docking?"* is really **two** questions, and
they are answered by two different experiments:

| the question | what answers it |
|---|---|
| **is the pose right?** | redocking, as in part 1 — and the checks below |
| **is the ranking right?** | whether it separates compounds known to work from compounds known not to |

Passing one does not pass the other, and this is not a technicality. In this very system
Attracting Cavities puts the quinazoline within 0.3 Å of its crystal position every time, and
still gives you no licence to say which of two compounds binds more tightly.

## The order you do it in

Cheapest first. You stop climbing when the evidence is strong enough for the claim you want to
make — not when you run out of things to try.

**1. Run it again.** Same inputs, different random seed, several times. If the top pose moves, you
have not found an answer, you have found one sample from a distribution.

<details>
<summary>🔍 What to look at in the repeats</summary>

Group the poses by how similar they are to each other. A tight, crowded group is worth much more
than a single good score, because it means the search keeps arriving at the same place. A set of
poses scattered all over the pocket means the search has not settled, and whichever one came out
on top did so by luck.

You can also plot score against distance-from-the-best-pose. What you hope to see is a funnel:
the closer a pose is to the best one, the better it scores. No funnel means the scoring function
is not pointing anywhere in particular.

Part 1 did a small version of this for you: two identical Attracting Cavities runs, where
everything about the ranking moved and only the ring system stayed put.

</details>

**2. Look at the geometry.** Not the score — the structure. Most wrong poses look wrong.

<details>
<summary>🔍 The usual tells</summary>

* **A buried polar group with nothing to pair with.** A nitrogen or oxygen that can donate or
  accept a hydrogen bond, buried in the protein with no partner, is the single strongest sign a
  pose is wrong. Real binding modes satisfy their polar groups or leave them in the water.
* **A charge with no counter-charge**, buried in a greasy pocket.
* **Greasy groups pointing into solvent** while polar ones are buried — the wrong way round.
* **Strained internal geometry**: a twisted amide, an eclipsed bond, atoms of the ligand too close
  to each other. The program was allowed to bend your molecule; check what it did.
* **A pose pressed against the edge of the search box.** That is a setup artefact, not a result.
  Re-run with a bigger box.
* **The protonation state you actually docked.** Often not the one you assumed, and it changes the
  electrostatics of the part that binds.

</details>

**3. Ask whether the pose explains what you already know.** This is the most valuable check in the
whole list, and the one people skip.

<details>
<summary>🔍 Why this one carries so much weight</summary>

You almost always have more than one compound, with measured activities. A correct binding mode
has to explain *why* one substituent kills activity and another does not.

So: find a change in your series that matters a lot, and look at where that part of the molecule
sits in your pose. If a group the data say is essential is pointing into bulk solvent, touching
nothing, the pose is wrong — however well it scored. If a group the data say can be anything is
buried against the protein, the pose is probably wrong too.

The same applies to knowledge from the literature rather than your own data. 4-anilinoquinazoline
kinase inhibitors are known to hydrogen-bond through the quinazoline N1 to the hinge backbone NH —
Met793 in EGFR. A pose that misses that interaction is suspect no matter what the number says.

This check needs no structure of your complex at all. It only needs activities you already have.

</details>

**4. Ask a second program, and a second structure.** You did the first half in part 2.

<details>
<summary>🔍 And why agreement proves less than it feels like it does</summary>

Two programs agreeing on a pose is weak positive evidence. Two programs disagreeing is strong
evidence that at least one is wrong. The asymmetry is the useful part.

It is only weak evidence because the programs are not independent in the way you would like. They
share the same big approximations — a rigid protein, no explicit water, a scoring function fitted
to the same public data — so they can be wrong together, in the same direction, for the same
reason.

The second structure matters as much. Dock into several structures of the same protein, or into
snapshots from a molecular dynamics run, and see whether your pose survives. A pose that only
appears when one side chain happens to be in one particular position is a fragile hypothesis.

</details>

**5. Let the pose move.** Docking holds the protein rigid. Releasing that is the first physical
test the pose has faced.

<details>
<summary>🔍 Minimisation, rescoring, and a short simulation</summary>

Start by letting the complex relax into its nearest energy minimum. If the ligand slides 3 Å in
the process, the pose was never a minimum to begin with.

Then run a short molecular dynamics simulation of the complex — the kind you analysed in notebook
5 — and ask two questions. Does the ligand stay where you put it, or drift out? And do the
hydrogen bonds you care about persist, measured as the fraction of the simulation they are present
for rather than in a single snapshot?

Treat this as a filter, not a proof: a wrong pose can sit still for a long time. It is much better
at telling you a pose is bad than at telling you it is right.

Beyond that lies proper free-energy calculation, which can rank compounds in a series far better
than any docking score — and costs enormously more.

</details>

**6. Test it in the lab.** A pose is a prediction about a physical object, so make the measurement
it predicts.

<details>
<summary>🔍 The experiments that attack a pose most directly</summary>

* **Mutate a contact residue.** Your pose names the residues it touches. If changing one of them
  does not change the affinity, the pose is wrong. This is the most direct test there is, and it
  needs no new chemistry.
* **Make the compound the pose suggests.** If the pose shows an unfilled space beside your
  molecule, add a group that fills it and measure what happens. A prediction that comes true is
  worth more than any amount of rescoring.
* **Measure the affinity properly** (ITC, SPR) rather than inferring it from an assay readout, and
  **check the binding is competitive** with whatever you think it competes with.
* **Locate the site without guessing** — NMR chemical-shift changes on the protein, for instance,
  tell you which part of it the compound is near.
* **Solve the structure.** This is the real answer, and note where it sits in the list: usually a
  result of the project, not an input to it.

</details>

## When there is no crystal structure to compare against

This is the normal case, and it is worth being precise about what is missing. You have no
structure *of your complex*. You almost always still have some experimental constraint — and the
job is to go and find the one you do have, then build your control around it:

* **A known active for your target.** Dock it. Does the protocol reproduce the binding mode it is
  known to have, and does the pose explain its SAR? If it fails there, it has no credibility on
  your compound either.
* **A related protein that *does* have a structure with a ligand in it.** Run part 1's redocking
  control there. The verdict transfers to the family, with that caveat stated out loud.
* **Published mutagenesis.** Somebody has often already shown which residues matter. Your pose
  should touch them.
* **Your own activity series.** Even with no structure anywhere, this supports check 3 — and ranks
  as evidence above anything the scoring function says.

If you genuinely have none of these, then say so. A pose with no experimental anchor is a
hypothesis worth generating and not evidence worth citing, and the distinction is the whole point
of this week.

<details>
<summary>🔍 One extra warning if your protein is a predicted model</summary>

If you are docking into a homology model or an AlphaFold structure rather than a crystal
structure, add a layer of doubt. Backbones in good models are often excellent; the side chains
lining a pocket are much less reliable, and they are exactly what docking depends on.

Worse, the model may be in the wrong *state*. Many proteins have an open and a closed form, or an
active and an inactive one, and a predicted model gives you one of them without telling you which.
Docking a ligand that needs the other one cannot work.

So before trusting anything: check the model's pocket can accommodate a ligand already known to
bind that protein.

</details>

## How strong a claim can you make?

Match the claim to the evidence you actually have. These are three different claims, not three
ways of saying one thing:

| the claim | what it needs |
|---|---|
| "this compound probably binds in this pocket, roughly here" | a redocking control that passed |
| "the core sits here, and makes this specific contact" | core RMSD validated, consistent with your SAR, stable when it is allowed to move |
| "compound A binds ten times more tightly than B" | **docking does not support this** — a free-energy calculation, or an experiment |

Deciding which of those you need *before* you run the docking is most of the skill. Picking which
twenty of two thousand compounds to buy is a perfectly good use of a protocol that is only
moderately accurate, because you need it to be better than random and the assay does the rest.
Asserting a binding mechanism from the same output is a far stronger claim on the same evidence.

Which brings it back to the three sentences you just wrote. The last one — what would change your
mind — is the one that decides whether everything above was science.

# What this week showed

| | |
|---|---|
| **a docking score is not an affinity** | different units, different meaning, and a correlation with measured values that is weak at best |
| **a pose is a hypothesis with an error bar** | and the redocking control is how you measure that error bar |
| **one RMSD hides where a pose is wrong** | core versus periphery changed the verdict entirely |
| **two programs disagreeing is information** | it bounds how much either one can be trusted |
| **a contact map answers "could it be steric?"** | not "is this the mechanism?" |

Four methods have now been pointed at the same molecule: a measurement, a model that knows only
substructures, a model that knows the pocket but not the solvent, and a model that has read every
structure ever deposited. They disagree by one to two log units, and each is wrong in a way the
others are not.

That is not a failure of the field. It is why a drug discovery programme runs several of them and
then makes the compound.